In [1]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus



In [2]:
datafile = r"C:\Users\affan\customer_shopping_behavior.csv"

In [3]:
df  = pd.read_csv(datafile)

In [4]:
df.head()


,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [5]:
df.describe() ##summary

,Customer ID,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.000000,3900.000000,3900.000000,3863.000000,3900.000000
mean,1950.500000,44.068462,59.764359,3.750065,25.351538
std,1125.977353,15.207589,23.685392,0.716983,14.447125
min,1.000000,18.000000,20.000000,2.500000,1.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000


In [6]:
df.isnull().sum()  ##checking null values

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [7]:
# to take care of the null values, we'll group it by category because there're different products. 
# We can obviously use the median values for all the products, but that'll create a bias. 
# So we'll create categories and then apply median to take care of the null values.

In [8]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [9]:
df.isnull().sum()   #Check the null values

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [10]:
# Now if you look at the column name, they're a mix of upper and lower case and spaces. For analysis in later 
# stages,we need uniformity, so we'l change the names and replace spaces with underscores

In [11]:
df.columns = df.columns.str.lower()    #change the column names to lower cases
df.columns = df.columns.str.replace(' ' , '_')   #replace spaces with underscores

In [12]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount_(usd)', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [13]:
# now only the purchase amount column looks different, so we'll remove the usd from it
df = df.rename(columns = {'purchase_amount_(usd)' : 'purchase_amount'})
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [14]:
# Now we'll categorize the age groups for better understanding. This is very important 
# for understanding customer behaviour

labels = ['Young adult' , 'Adult' , 'Middle-aged' , 'Senior']
df['age_group'] = pd.qcut(df['age'] , q=4 , labels = labels)

In [15]:
df[['age' , 'age_group']].head(10)

,age,age_group
0,55,Middle-aged
1,19,Young adult
2,50,Middle-aged
3,21,Young adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young adult
8,26,Young adult
9,57,Middle-aged


In [16]:
# To understand when th ecustomer is making purchases in what frequency is important for 
# customer behaviour and shopping habits

# Create a dictionary 

frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)


In [17]:
df[['purchase_frequency_days' , 'frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [18]:
# Now we'll check if the dicounts applied and promo code applied columns are redundant,
# if they are, we'll remove them

(df['discount_applied'] == df['promo_code_used']).all()

True

In [19]:
df = df.drop('promo_code_used', axis=1)  #dropping the promo code used column

In [20]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='object')

In [21]:
# Now we connect to the MS SQL Server

In [24]:
!pip install pyodbc pandas sqlalchemy

In [22]:
import pyodbc
import pandas as pd

server = 'localhost\SQLEXPRESS'
database = 'customer_behaviour'

conn_str = (
    'DRIVER={ODBC Driver 18 for SQL Server};'
    f'SERVER={server};'
    f'DATABASE={database};'
    'Trusted_Connection=yes;'
    'Encrypt=yes;'
    'TrustServerCertificate=yes;'
)

conn = pyodbc.connect(conn_str)

print("Connected successfully!")

Connected successfully!


<>:4: SyntaxWarning: invalid escape sequence '\S'
<>:4: SyntaxWarning: invalid escape sequence '\S'
C:\Users\affan\AppData\Local\Temp\ipykernel_7908\1906481798.py:4: SyntaxWarning: invalid escape sequence '\S'
  server = 'localhost\SQLEXPRESS'


In [23]:
print(pd.read_sql("SELECT DB_NAME() AS DatabaseName", conn))

         DatabaseName
0  customer_behaviour


C:\Users\affan\AppData\Local\Temp\ipykernel_7908\480225837.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql("SELECT DB_NAME() AS DatabaseName", conn))


In [24]:
# # Create a SQLAlchemy connection

from sqlalchemy import create_engine
from urllib.parse import quote_plus

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + quote_plus(conn_str)
)

In [25]:
# Upload your DataFrame to SQL Server

df.to_sql(
    name='customer',         #  Creates a table called customer
    con=engine,
    schema='dbo',
    if_exists='replace',     #  Creates the table or replaces an existing one
    index=False              #	Does not upload the Pandas index
)

print("Data successfully uploaded to SQL Server!")

D:\anaconda\Lib\site-packages\pandas\io\sql.py:1636: SAWarning: Unrecognized server version info '17.0.1155.3'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Data successfully uploaded to SQL Server!


In [26]:
# Verify that your data is in SQL Server

query = """
SELECT TOP 10 *
FROM dbo.customer
"""

sql_data = pd.read_sql(query, conn)

display(sql_data)

C:\Users\affan\AppData\Local\Temp\ipykernel_7908\525367085.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sql_data = pd.read_sql(query, conn)


,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle-aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle-aged,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young adult,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Middle-aged,365
5,6,46,Male,Sneakers,Footwear,20,Wyoming,M,White,Summer,2.9,Yes,Standard,Yes,14,Venmo,Weekly,Middle-aged,7
6,7,63,Male,Shirt,Clothing,85,Montana,M,Gray,Fall,3.2,Yes,Free Shipping,Yes,49,Cash,Quarterly,Senior,90
7,8,27,Male,Shorts,Clothing,34,Louisiana,L,Charcoal,Winter,3.2,Yes,Free Shipping,Yes,19,Credit Card,Weekly,Young adult,7
8,9,26,Male,Coat,Outerwear,97,West Virginia,L,Silver,Summer,2.6,Yes,Express,Yes,8,Venmo,Annually,Young adult,365
9,10,57,Male,Handbag,Accessories,31,Missouri,M,Pink,Spring,4.8,Yes,2-Day Shipping,Yes,4,Cash,Quarterly,Middle-aged,90


In [27]:
query = """
SELECT COUNT(*) AS Total_Records
FROM dbo.customer
"""

pd.read_sql(query, conn)

C:\Users\affan\AppData\Local\Temp\ipykernel_7908\935206291.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query, conn)


,Total_Records
0,3900


In [60]:
# Close SQLAlchemy connection pool
#engine.dispose()

# Close pyodbc connection
#conn.close()

#print("Database connections closed successfully.")

Database connections closed successfully.
